# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [9]:
# 1
len(clean)

5463

In [14]:
sep = amount[clean & (month == "2026-09")].sum()
octob = amount[clean & (month == "2026-10")].sum()
change = (rev_oct - rev_sep) / rev_sep * 100

print(f"Продажи сентябрь:{sep}, Продажи октябрь:{octob}, Разница:{round(change, 2)} %")

Продажи сентябрь:35243410.0, Продажи октябрь:34419520.0, Разница:-2.34 %


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2

In [23]:
cats = np.unique(cat)
months = np.unique(month)  

table = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                  for c in cats])

for c, r in zip(cats, table):
    print(f"{c:<15} {r[0]:>12,.0f} {r[1]:>12,.0f}")
print(table.shape)   


Дом                4,997,730    5,125,790
Книги              1,065,300    1,140,910
Красота            2,610,950    2,678,990
Одежда             9,167,100    8,505,730
Электроника       17,402,330   16,968,100
(5, 2)


In [24]:
ttotal = table.sum(axis=1)
best = cats[total.argmax()]
print("Больше всего выручки:", best, f"{total.max():,.0f}")

Больше всего выручки: Электроника 93,904,758,160


In [25]:
change = (table[:, 1] - table[:, 0]) / table[:, 0] * 100
for c, ch in zip(cats, change):
    print(f"{c:<15} {ch:>7.2f} %")
worst = cats[change.argmin()]
print("Сильнее всех упала:", worst, f"{change.min():.2f} %")

Дом                2.56 %
Книги              7.10 %
Красота            2.61 %
Одежда            -7.21 %
Электроника       -2.50 %
Сильнее всех упала: Одежда -7.21 %


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3

In [26]:
counts = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                   for c in cats])

avg_check = table / counts 

In [27]:
for c, r in zip(cats, avg_check):
    print(f"{c:<15} {r[0]:>10,.0f} {r[1]:>10,.0f}")

Дом                 10,499     10,768
Книги                4,020      3,816
Красота              6,187      6,159
Одежда              13,521     13,046
Электроника         32,773     33,600


In [28]:
best = cats[avg_check[:, 1].argmax()]
print("Самый высокий средний чек в октябре:", best, f"{avg_check[:, 1].max():,.0f}")

Самый высокий средний чек в октябре: Электроника 33,600


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4

In [38]:
cities = np.unique(city)

table_city = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                       for c in cities])

share = table_city / table_city.sum(axis=0)
for c, r in zip(cities, share):
    print(f"{c:<10} {r[0]:>7.2%} {r[1]:>7.2%}")

Актобе       7.31%   7.38%
Алматы      41.15%  42.54%
Астана      25.63%  25.71%
Атырау       2.62%   2.82%
Караганда   10.35%   9.80%
Шымкент     12.93%  11.75%


In [39]:
print(share.sum(axis=0))                  

[1. 1.]


In [40]:
print("Шымкент:", share[5])
print("Изменение:", share[5, 1] - share[5, 0])

Шымкент: [0.12934759 0.11749176]
Изменение: -0.011855825342961729


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

##### table имеет форму (6, 2), а table.sum(axis=1) даёт массив формы (6,). NumPy не может их поделить: формы (6, 2) и (6,) несовместимы, потому что последние размеры 2 и 6 не совпадают.

In [42]:
share = table / table.sum(axis=1, keepdims=True)

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8